## Proyecto: Iberdrola DTR

Create and populate a new table ucgeproc from geproc adding two columns: ucIEEE738 .- DLR computed using the IEEE738 approach developed by the UC; ucCIGRE601 .- DLR computed using the CIGRE601 approach developed by the UC.

Autor: GTEA. Universidad de Cantabria   
Última revisión: 3/10/2023


**Paquetes generales**

In [14]:
import psycopg2 # Database manipulation
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date 
from dateutil.relativedelta import relativedelta
from tqdm.notebook import trange, tqdm # Progress bar
from time import sleep
import warnings
import xlrd # Excel manipulation
import configparser # Config file
import matplotlib.pyplot as plt
import plotly.express as px
from dash import dcc
import calendar # operation with dates
import pyodbc
import time
from sqlalchemy import create_engine



# conda activate the environment_you_intend_to_use
# pip install --upgrade --quiet jupyter_client ipywidgets

In [3]:
import sys
print( sys.version)

3.9.15 (main, Nov 24 2022, 14:39:17) [MSC v.1916 64 bit (AMD64)]


**Paquetes específicos DLR**

In [4]:
from cable import cable
from case import case
from ieee738 import ieee738
from cigre601 import cigre601
from pvsystems import pvsystems
import matplotlib.pyplot as plt 


# Needed only during the development phase.
from importlib import reload
reload( cable)
reload( case)
reload( ieee738)
reload( cigre601)
reload( pvsystems)

<module 'pvsystems.pvsystems' from 'e:\\mario\\python\\pypacity\\pvsystems\\pvsystems.py'>

**DATOS**

Conductor: LA-280

In [5]:

NSELECT = 2 
CablePE = cable.Cable()
c_db, error = CablePE.load_cable_db()
CablePE.EMISS = 0.8
CablePE.ABSORP = 0.8

PV1 = pvsystems.PVSystems()
Case1 = case.Case()
Case1.demo( NSELECT)
# Ambient conditions
Case1.TAMB = 40.0
Case1.CDR_LAT_DEG = 30
Case1.ALBEDO = 0.1
Case1.beta = 0
Case1.CDR_ELEV = 0
Case1.TCDRPRELOAD = 100
#Case1.TCDRMAX = 150
#Case1.TCDR = 100.0
Case1.WINDANG_DEG = 60
Case1.Z1_DEG = 90
Case1.Ns = 1.0
Case1.SUN_TIME = 11
Case1.NDAY = PV1.DayOfYear( 10, 6) # 10th June
#print("NDAY: " + str(Case1.NDAY))

NDAY: 161


Read config parameters from config file "config.ini"

In [6]:
ConfigFile = r".\\config.ini"
configP = configparser.ConfigParser()
configP.read(ConfigFile)

['.\\\\config.ini']

In [7]:
PWD = configP['GENERAL']['PWD']
USER = configP['GENERAL']['USER']
DATABASE = configP['GENERAL']['DATABASE']


In [8]:
def GetNumberValuesMonth( iYear, iMonth, deltaT=5):
    firstday, lastday = calendar.monthrange( iYear, iMonth)
    if np.isnan( deltaT) == 0:
        Nvalues = int(lastday*24*60/deltaT)
    else:
        Nvalues = 0
    return Nvalues


def GetDatesFromYearMonth( iYear, iMonth):
    firstday, lastday = calendar.monthrange( iYear, iMonth)
    firstdate =datetime( iYear, iMonth, 1, hour=0, minute=0, second=0, microsecond=0, tzinfo=None)
    lastdate = datetime( iYear, iMonth, lastday, hour=23, minute=59, second=0, microsecond=0, tzinfo=None)
    #print( firstdate)
    #print( lastdate)
    return firstdate, lastdate


def FilterByDates( DFdata, firstdate, lastdate):
    DFdatafiltered = DFdata.loc[(DFdata['measurementTime'] >= firstdate) & (DFdata['measurementTime'] <= lastdate)]
    return DFdatafiltered


def FilterByNodeID( DFdata, NID):
    DFdatafiltered = DFdata.loc[ DFdata['nodeId'] == NID]
    return DFdatafiltered


def GenerateDatesBetween( firstdate, lastdate):
    fechas = []
    fechasdt = []
    fecha_actual = firstdate

    while fecha_actual <= lastdate:
        fecha_actual_str = fecha_actual.strftime("%Y-%m")
        fechas.append( fecha_actual_str)
        fechasdt.append( datetime.strptime( fecha_actual_str, "%Y-%m"))
        fecha_actual += relativedelta(months=1)

    return fechasdt


def AnalyzeDataByMonth( DFdata, firstdate, lastdate, NodeID):
    dateslist = GenerateDatesBetween( firstdate, lastdate)
    NValT = 0
    NRealValT = 0
    for data in dateslist:
        iyear = data.year
        imonth = data.month
        fd, ld = GetDatesFromYearMonth( iyear, imonth)
        DFf = FilterByDates( rDF, fd, ld)
        DFf = FilterByNodeID( DFf, NodeID)
        deltaT = GetDeltaT( DFf, fd, ld, NodeID)
        NVal = GetNumberValuesMonth( iyear, imonth, deltaT/60)
        NRealVal = len(DFf)
        NValT += NVal
        NRealValT += NRealVal
        print('year: ', iyear, ' ; month: ', imonth, ' ; valores teóricos: ', NVal, ' ; valores medidos: ', NRealVal)
    print("Node: " + str(NodeID) + " / Valores Teóricos: " + str(NValT) + " / Valores medidos: " + str(NRealValT))


def GetDeltaT( DFdata, firstdate, lastdate, NodeID):
    ElPalmarNodeID = [ '10160',  '10166', '10174']
    HellinNodeID = [ '10021', '10031', '10042', '10055', '10068', '10085']
    if NodeID in ElPalmarNodeID:
        return( 60)
    if NodeID in HellinNodeID:
        return( 300)
    
    DeltaTs = []
    dateslist = GenerateDatesBetween( firstdate, lastdate)
    for data in dateslist:
        iyear = data.year
        imonth = data.month
        NVal = GetNumberValuesMonth( iyear, imonth)
        fd, ld = GetDatesFromYearMonth( iyear, imonth)
        DFf = FilterByDates( rDF, fd, ld)
        DFf = FilterByNodeID( DFf, NodeID)
        DFf = DFf[ DFf['ambientTemperature'] != 'NaN']
        if len(DFf)>0:
            d1 = DFf['measurementTime']
            t1 = d1.iloc[0]
            t2 = d1.iloc[1]
            dt = t2-t1
            dt = dt.total_seconds()
            DeltaTs.append( dt)
    if len(DeltaTs)>0:
        AVGDeltaTs = np.average( DeltaTs)
    else:
        AVGDeltaTs = 0
    return( AVGDeltaTs)
                  

In [9]:
DATABASE

'Iberdrola'

Connection to the table geproc in the database Iberdrola

In [10]:
# Conectar con la BBDD
database = psycopg2.connect( database = DATABASE, user = USER, password = PWD)
database.autocommit = True
cursor = database.cursor() # cursor to the geproc database [source]
cursoruc = database.cursor() # cursor to the ucgeproc database [target]


In [11]:
# Create a cursor object to interact with the database
#cursor = conn.cursor()

# Get column names from the specified table
query = f"SELECT column_name FROM information_schema.columns WHERE table_name = 'geproc';"
cursor.execute(query)

# Fetch all column names
column_names = [row[0] for row in cursor.fetchall()]

# Close the cursor and connection
#cursor.close()
#conn.close()

# Print the column names
for column in column_names:
    print(column)

#Make sure to replace the placeholders in the script with your actual PostgreSQL database credentials and the table name from which you want to retrieve the column names.
#This script connects to the PostgreSQL database, queries the information_schema to retrieve column names for the specified table, and then prints the column names.







MaxCapacityMVA
Clearance
IMAX
LoadMVA
geprocid
PhaseCurrent
PhasePhase
ConductorTemp
AmbientTemp
WindSpeed
WindDomDirection
WindAvgDirection
SolarRadiation
DewPoint
LineName
NodeName
TimeStamp
ServiceName


In [17]:
column_names = [#"geprocid",
"LineName",
"NodeName",
"TimeStamp",
"PhaseCurrent",
"PhasePhase",
"ConductorTemp",
"AmbientTemp",
"WindSpeed",
"WindDomDirection",
"WindAvgDirection",
"SolarRadiation",
"DewPoint",
"ServiceName",
"Clearance",
"IMAX",
"LoadMVA",
"MaxCapacityMVA",
"IEEE738",
"CIGRE601",
"ucIEEE738",
"ucCIGRE601"]


ucgeproc = pd.DataFrame( columns = column_names)
print(ucgeproc)

Empty DataFrame
Columns: [LineName, NodeName, TimeStamp, PhaseCurrent, PhasePhase, ConductorTemp, AmbientTemp, WindSpeed, WindDomDirection, WindAvgDirection, SolarRadiation, DewPoint, ServiceName, Clearance, IMAX, LoadMVA, MaxCapacityMVA, IEEE738, CIGRE601, ucIEEE738, ucCIGRE601]
Index: []

[0 rows x 21 columns]


In [18]:
create_ucgeproc = True
# PostgreSQL connection parameters
db_params = {
    'host': 'localhost',
    'database': 'Iberdrola',
    'user': 'postgres',
    'password': 'mananam05'
}

# Establish a connection to the PostgreSQL database using SQLAlchemy
connection_string = f"postgresql+psycopg2://{db_params['user']}:{db_params['password']}@{db_params['host']}/{db_params['database']}"
engine = create_engine(connection_string)

# Define the name of the table you want to create or replace
table_name = 'ucgeproc'


if create_ucgeproc == True:
    ucgeproc.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid') 


In [ ]:
# Plantilla SQL para leer datos de la BBDD Iberdrola, tabla geproc
sqlstr = '''SELECT * FROM public.geproc
WHERE geproc."AmbientTemp" IS DISTINCT FROM 'NaN' 
AND geproc."WindSpeed" IS DISTINCT FROM 'NaN'
AND geproc."SolarRadiation" IS DISTINCT FROM 'NaN'
order by geproc."TimeStamp" ASC;'''

# WHERE linename = 'Hellin-Calasparra' 
# -- where measurementtime > '2021-12-01 00:06:00' and measurementtime < '2021-12-01 00:10:00'

In [ ]:
cursor.execute( sqlstr)
results = cursor.fetchall()

In [ ]:
type( results)

In [ ]:
rDF = pd.DataFrame(results)
columnsDF = ['geDTRId',
             'siteName',
             'lineName',
             'nodeId',
             'measurementTime',
             'ambientTemperature',
             'windSpeed',
             'windSpeedavg',
             'windDirection',
             'windDirectionD',
             'solarRadiation',
             'dewPoint',
             'windSonicWindSpeed',
             'windSonicWindDirectionavg',
             'windSonicWindDirectionD',
             'conductorTemperature',
             'conductorCurrent',
             'quality']
rDF.columns = columnsDF

In [ ]:
rDF

Close the connection

In [ ]:
cursor.close()

Number of elements

In [ ]:
print(type(  rDF['measurementTime']))
start = datetime(  year=2021, month=12, day=1, hour=1, minute=0, second=0, microsecond=0, tzinfo=None)
end = datetime( year=2022, month=12, day=31, hour=23, minute=55, second=0, microsecond=0, tzinfo=None)
deltat =timedelta( minutes=5)

currenttime = start
timelist = []

while currenttime < end:
    timelist.append( currenttime)
    currenttime += deltat

In [ ]:
print( len(timelist))


In [ ]:
imprimirfigura = 0

if imprimirfigura == 1:
    fig = px.scatter( rDF, x='measurementTime', y='ambientTemperature', color='nodeId')
    fig.update_traces( mode='markers+lines')
    fig.show()

### Obtener el rango temporal de datos en la BBDD

In [ ]:
listafechas = rDF['measurementTime']
Primerdato = min(listafechas)
Ultimodato = max(listafechas)
print('Primer dato: ', Primerdato)
print('Último dato: ', Ultimodato)

In [ ]:
type( rDF['measurementTime'])

In [ ]:
ElPalmarNodeID = [ '10160',  '10166', '10174']
HellinNodeID = [ '10021', '10031', '10042', '10055', '10068', '10085']

In [ ]:
type(rDF['nodeId'])

In [ ]:
for nodes in ElPalmarNodeID:
    d1 = GetDeltaT( rDF, Primerdato, Ultimodato, nodes)
    print(d1)

In [ ]:
for nodes in HellinNodeID:
    d1 = GetDeltaT( rDF, Primerdato, Ultimodato, nodes)
    print(d1)

In [ ]:
print('Línea El Palmar - Espinardo')
for nodes in ElPalmarNodeID:
    print('NodeID: ', nodes)
    AnalyzeDataByMonth( rDF, Primerdato, Ultimodato, nodes)
print('*****************************************')

print('Línea Hellín - Calasparra')
for nodes in HellinNodeID:
    print('NodeID: ', nodes)
    AnalyzeDataByMonth( rDF, Primerdato, Ultimodato, nodes)
print('*****************************************')